In [ ]:
#!/bin/bash
!curl -L https://www.kaggle.com/api/v1/datasets/download/abdurrahman22224/smartphone-new-data -o data.zip --skip-existing
!unzip -o data.zip'
!ls -lah

In [ ]:
import os
from pyspark.sql import SparkSession

In [ ]:
catalog_name = "nessie"
namespace = "smartphone"
table_name = "usage"
table_ident = f"{catalog_name}.{namespace}.{table_name}"

iceberg_warehouse = "s3://test/sources/"
nessie_catalog_uri = "https://sources.k8s-jagr.duckdns.org/iceberg"
s3_endpoint_host = "https://garage.k8s-jagr.duckdns.org"
s3_region = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
aws_access_key = "GK0123456789abcdef01234567"
aws_secret_key = "0123456789abcdef0123456789abcdef0123456789abcdef0123456789abcdef"

In [ ]:
packages = ",".join([
    "org.apache.hadoop:hadoop-aws:3.4.2",
    "software.amazon.awssdk:bundle:2.25.53",
    "software.amazon.awssdk:url-connection-client:2.25.53",
    "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1",
])

spark = (
    SparkSession.builder
    .appName("smartphone-to-iceberg-compose")
    .config("spark.jars.packages", packages)
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.access.key", aws_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", aws_secret_key)
    .config("spark.hadoop.fs.s3a.endpoint", s3_endpoint_host)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", s3_region)
    .config(f"spark.sql.catalog.{catalog_name}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog_name}.type", "rest")
    .config(f"spark.sql.catalog.{catalog_name}.uri", nessie_catalog_uri)
    .config(f"spark.sql.catalog.{catalog_name}.warehouse", iceberg_warehouse)
    .config(f"spark.sql.catalog.{catalog_name}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{catalog_name}.s3.endpoint", f"{s3_endpoint_host}")
    .config(f"spark.sql.catalog.{catalog_name}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{catalog_name}.s3.region", s3_region)
    .getOrCreate()
)

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./smartphone_cleaned_v5.csv")
)

print("Rows in source:", df.count())
df.printSchema()
df.show(10, truncate=False)

In [ ]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS nessie.smartphone")

if not spark.catalog.tableExists(table_ident):
    (
        df.limit(0)
        .writeTo(table_ident)
        .using("iceberg")
        .tableProperty("write.format.default", "parquet")
        .create()
    )
    print("Created table:", table_ident)
else:
    print("Table already exists:", table_ident)

In [ ]:
df.writeTo(table_ident).append()